# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Refresh Prediction lane — warehouse data contract, verified with queries on `month=2026-03`.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-data-contracts` + `flyrank/flyrank-data`.

In [1]:
%pip -q install duckdb huggingface_hub truststore scikit-learn

import os
import sys

import truststore
truststore.inject_into_ssl()

IN_COLAB = "google.colab" in sys.modules
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    from pathlib import Path
    local = Path("../.hf_token_local")
    if local.is_file():
        HF_TOKEN = local.read_text(encoding="utf-8").strip()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing — use Colab Secret HF_TOKEN, shell env var, or work/.hf_token_local (local)."
    )

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_march": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    "fact_daily_feb_march": (
        "read_parquet(["
        f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet',"
        f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
        "])"
    ),
}
# confirm sealed final month exists (not used for labels/features)
_sample_n = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')").fetchone()[0]
print(f"Sealed _sample table rows (June 2026 only): {_sample_n:,} — excluded from this contract.")


Note: you may need to restart the kernel to use updated packages.


Sealed _sample table rows (June 2026 only): 11,694,072 — excluded from this contract.


## Step 0 — Schema discovery (before any contract text)

Enumerated subsets and columns from the hosted release:

In [2]:
import pandas as pd

schema_rows = []
for name, src in TABLES.items():
    if name == "fact_daily_feb_march":
        continue
    cols = con.sql(f"DESCRIBE SELECT * FROM {src}").df()
    for _, row in cols.iterrows():
        schema_rows.append(
            {"subset": name, "column": row["column_name"], "dtype": row["column_type"]}
        )
schema_df = pd.DataFrame(schema_rows)
subset_summary = (
    schema_df.groupby("subset")
    .agg(n_columns=("column", "count"), example_columns=("column", lambda s: ", ".join(list(s)[:6])))
    .reset_index()
)
print(subset_summary.to_string(index=False))
schema_df.head(20)

          subset  n_columns                                                                                        example_columns
     dim_clients          9         client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile, client_created_date
     dim_content         26 client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, keyword_char_count, keyword_token_count
fact_daily_march         31       report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available


,subset,column,dtype
0,dim_clients,client_hash_id,VARCHAR
1,dim_clients,is_active,BOOLEAN
2,dim_clients,has_gsc_access,BOOLEAN
3,dim_clients,has_ga4_access,BOOLEAN
4,dim_clients,access_profile,VARCHAR
5,dim_clients,client_created_date,DATE
6,dim_clients,client_updated_date,DATE
7,dim_clients,gsc_data_start,DATE
8,dim_clients,ga4_data_start,DATE
9,dim_content,client_hash_id,VARCHAR


## 1. Unit of analysis + time window

**What one row means (Refresh Prediction lane).** One row is one pseudonymized page (`content_hash_id`) under one client (`client_hash_id`), scored at the **decision date** `2026-03-31` for “refresh this week” prioritisation — not one calendar day. Daily facts are rolled into trailing 30-day / previous-30-day windows that end on that date; the ranking unit stays page × client, matching `w02_ml_task_framing.ipynb`.

**Which table(s).** `fact_content_daily_performance` (partitions `month=2026-02` and `month=2026-03` for trailing windows) joined to `dim_content` for static page context (`content_created_date`, `word_count`, `content_type`). `dim_clients` is available for coverage checks but not required for this slice.

**Time window.** Development month **`month=2026-03`** (mid-panel: not the sealed final month `2026-06` in `fact_content_daily_performance_sample`). Decision date = last day in that partition (`2026-03-31`). February is read only to compute the previous-30-day impression leg; no label logic is fit on June 2026.

**What we predict or rank (from w02 — not redefined).** Learning-to-rank / priority scoring: order pages by **expected recoverable impressions at stake** using decline-like movement (`trend_direction == "down"` as the decline proxy), exposure (`impressions` scale), and CTR/position headroom. Success metric remains **Precision@50** vs the hand baseline under client-holdout in later weeks; here we only sanity-check a depth-2 tree on the March slice.

**One deliberate exclusion.** **`fact_content_daily_performance_sample`** (June 2026) — it is the panel’s terminal month, i.e. the natural outcome window for any past→future refresh label. We query it only to confirm it exists, never for features, labels, or the leakage demo.

## 2. Fields: feature / label / context / excluded

| Bucket | Fields |
|---|---|
| **Context (join/group only)** | `client_hash_id`, `content_hash_id`, `report_date` (daily fact only) |
| **Label / proxy** | `trend_direction`, `is_declining_label` (= `trend_direction == "down"`), `trend_pct` (label sibling — never a feature) |
| **Features (≤5 in section 3)** | `log_imp_prev30`, `imp_last30`, `avg_pos_last30`, `ctr_last30`, `content_age_days` (derived from `content_created_date`) |
| **Excluded** | `trend_pct`, `trend_direction` as model inputs; June `_sample` partition; any row with `gsc_data_available IS NOT TRUE` (zeros before tracking are not real zeros) |

## 3. Verify it with queries (grain, counts, availability)

All three queries run on the **`month=2026-03`** daily fact partition unless noted.

### Query 1 — Grain (daily fact)

In [3]:
q1 = con.sql(f'''
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM {TABLES['fact_daily_march']}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
''').df()
print(q1)
print("duplicate keys returned:", len(q1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []
duplicate keys returned: 0


*Proves the hosted daily fact grain is `report_date × client_hash_id × content_hash_id` — zero duplicate keys returned.*

### Query 2 — Slice row count and date span

In [4]:
q2 = con.sql(f'''
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {TABLES['fact_daily_march']}
''').df()
print(q2.to_string(index=False))

 row_count min_report_date max_report_date
   9841378      2026-03-01      2026-03-31


*Counts March-partition daily rows and shows the calendar span covered in this mid-panel slice.*

### Query 3 — Availability (`IS TRUE`)

In [5]:
q3 = con.sql(f'''
SELECT
    COUNT(*) AS rows_before,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_after_is_true
FROM {TABLES['fact_daily_march']}
''').df()
print(q3.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rows_before  rows_after_is_true
     9841378             3611061


*Filters with `gsc_data_available IS TRUE` (not `== True`); reports pre- and post-filter row counts.*

### Five-feature frame + leakage trap

Page-level frame for `month=2026-03` decision date (see feature notes below).

- `log_imp_prev30` — knowable at the decision moment because it sums GSC impressions strictly in the **previous** 30-day window ending 2026-03-31 (Feb–Mar daily facts only).
- `imp_last30` — knowable at the decision moment because it is measured GSC exposure in the trailing 30 days **ending on** the decision date, before any refresh action.
- `avg_pos_last30` — knowable at the decision moment because it averages `gsc_avg_position` over that same trailing 30-day pre-decision window.
- `ctr_last30` — knowable at the decision moment because it is clicks divided by impressions inside the trailing 30-day pre-decision window (rates on the warehouse 0–100 scale).
- `content_age_days` — knowable at the decision moment because it is `DATE_DIFF` from `content_created_date` to the decision date (`2026-03-31`) in `dim_content`, fixed before any refresh.

In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

DECISION_DATE = "2026-03-31"
pages = con.sql(f'''
    WITH daily AS (
        SELECT * FROM {TABLES['fact_daily_feb_march']}
        WHERE report_date <= DATE '{DECISION_DATE}'
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 60 DAY
                      AND report_date <= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN gsc_avg_position END) AS avg_pos_last30,
            BOOL_AND(gsc_data_available IS TRUE) AS gsc_ok
        FROM daily
        GROUP BY 1, 2
    ),
    labeled AS (
        SELECT
            a.*,
            CASE
                WHEN a.imp_prev30 = 0 AND a.imp_last30 > 0 THEN 'new'
                WHEN a.imp_prev30 = 0 AND a.imp_last30 = 0 THEN 'flat'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 > 20 THEN 'up'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 < -20 THEN 'down'
                ELSE 'stable'
            END AS trend_direction,
            CASE WHEN a.imp_prev30 = 0 THEN NULL
                 ELSE (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 END AS trend_pct
        FROM agg a
        WHERE a.imp_last30 > 0 AND a.gsc_ok IS TRUE
    )
    SELECT
        l.*,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days
    FROM labeled l
    LEFT JOIN {TABLES['dim_content']} d USING (content_hash_id)
    WHERE DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') >= 90
''').df()

pages["is_declining_label"] = pages["trend_direction"].eq("down").astype(int)
pages["ctr_last30"] = np.where(
    pages["imp_last30"] > 0, pages["clk_last30"] / pages["imp_last30"] * 100.0, 0.0
)
pages["log_imp_prev30"] = np.log1p(pages["imp_prev30"])

FEATURES = ["log_imp_prev30", "imp_last30", "avg_pos_last30", "ctr_last30", "content_age_days"]
feature_frame = pages[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
print(f"Page-level rows after filters: {len(pages):,}")
print(f"Decline proxy rate: {pages['is_declining_label'].mean():.3f}")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows after filters: 41,747
Decline proxy rate: 0.190


,log_imp_prev30,imp_last30,avg_pos_last30,ctr_last30,content_age_days
0,5.123964,146.0,5.133019,7.534247,228
1,6.177944,249.0,4.219047,2.409639,225
2,6.717805,1589.0,9.092964,0.000000,264
3,5.921578,407.0,7.913611,0.491400,188
4,6.850126,1156.0,7.097347,0.259516,188


In [7]:
# --- Leakage trap (deliberate) ---
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

y = pages["is_declining_label"].values
X_honest = feature_frame[FEATURES]

tree_leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
X_with_leak = X_honest.copy()
X_with_leak["trend_pct"] = pages["trend_pct"].fillna(0)
tree_leaky.fit(X_with_leak, y)
leaky_p50 = precision_at_k(tree_leaky.predict_proba(X_with_leak)[:, 1], y, 50)

tree_honest = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_honest.fit(X_honest, y)
honest_p50 = precision_at_k(tree_honest.predict_proba(X_honest)[:, 1], y, 50)

print(f"Leaky tree (5 features + trend_pct) Precision@50: {leaky_p50:.3f}")
print(f"Honest tree (5 features only) Precision@50:       {honest_p50:.3f}")

# remove leaky column — keep honest frame only
print("Final feature columns:", list(X_honest.columns))

Leaky tree (5 features + trend_pct) Precision@50: 1.000
Honest tree (5 features only) Precision@50:       0.600
Final feature columns: ['log_imp_prev30', 'imp_last30', 'avg_pos_last30', 'ctr_last30', 'content_age_days']


## 4. Data limits

**Named limitation — GSC availability gate on the March slice.** Requiring `gsc_data_available IS TRUE` on every day in the Feb–Mar window (via `BOOL_AND` before aggregation) keeps zero-filled pre-tracking rows out of the refresh queue, but it also drops pages whose GSC history is patchy or whose client started mid-window. The resulting page cohort is **biased toward clients with continuous Search Console coverage** in early 2026, so Precision@50 on this slice is a directional sanity check — not a guarantee for GSC-sparse clients or for the sealed June outcome month.

## Self-check

- Grain query returns **zero** duplicate `report_date × client × content` keys — it proves the daily fact grain rather than assuming it.
- **Exactly three** verification queries are shown above (grain, count/span, availability with `IS TRUE`).
- Availability uses **`IS TRUE`**, not truthiness or `== True`.
- Each of the five features has a **“knowable at the decision moment because…”** line.
- **`trend_pct`** was added deliberately, inflated Precision@50, then removed; the kept score is the **honest** five-feature tree.
- Committed under `work/notebooks/` — submit repo URL on the card.